In [ ]:
import re
import string
import numpy as np
import pandas as pd
import nltk
import spacy
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
import matplotlib.pyplot as plt
from collections import Counter
from typing import Tuple
from sklearn.linear_model import LogisticRegression


In [ ]:
train = pd.read_csv('/content/drive/MyDrive/Homework Team #2/NLP Final Project/train.csv', encoding='latin-1')
test = pd.read_csv('/content/drive/MyDrive/Homework Team #2/NLP Final Project/test.csv', encoding='latin-1')
attributes = pd.read_csv('/content/drive/MyDrive/Homework Team #2/NLP Final Project/attributes.csv', encoding='latin-1')
sample_submission = pd.read_csv('/content/drive/MyDrive/Homework Team #2/NLP Final Project/sample_submission.csv', encoding='latin-1')
product_descriptions = pd.read_csv('/content/drive/MyDrive/Homework Team #2/NLP Final Project/product_descriptions.csv', encoding='latin-1')

# 1. Handle missing values & format strings
attributes_clean = attributes.dropna(subset=['name', 'value']).copy()
attributes_clean['attr_text'] = attributes_clean['name'] + ': ' + attributes_clean['value']

# 2. Group by product_uid and join values with a separator
product_attributes = attributes_clean.groupby('product_uid')['attr_text'].apply(lambda x: ' | '.join(x)).reset_index()
product_attributes

df = (
    train.join(product_descriptions.set_index('product_uid'), on='product_uid')
    .join(product_attributes.set_index('product_uid'), on='product_uid')
    .drop(['product_uid', 'id'], axis=1)
)

df.head()


In [ ]:
# 1. Fill missing text fields safely across all text sources
text_cols = ['product_title', 'product_description', 'search_term', 'attr_text']
for col in text_cols:
  if col in df.columns:
    df[col] = df[col].fillna('').astype(str)

# ---------------------------------------------------------
# 1. Lengths, Word Counts, Ratios, Noise, & Digit Density
# ---------------------------------------------------------
# Character lengths
df['title_length'] = df['product_title'].str.len()
df['description_length'] = df['product_description'].str.len()
df['search_term_length'] = df['search_term'].str.len()
df['attr_length'] = df['attr_text'].str.len()

# Word counts
df['title_word_count'] = df['product_title'].str.split().str.len()
df['description_word_count'] = (
    df['product_description'].str.split().str.len()
)
df['search_term_word_count'] = df['search_term'].str.split().str.len()

# Title Noise Extraction (count & ratio of special non-alphanumeric characters)
df['title_noise_count'] = df['product_title'].str.count(r'[^a-zA-Z0-9\s]')
df['title_noise_ratio'] = df['title_noise_count'] / np.maximum(
    df['title_length'], 1
)

# Character Length Ratios
df['search_to_title_len_ratio'] = df['search_term_length'] / np.maximum(
    df['title_length'], 1
)
df['search_to_desc_len_ratio'] = df['search_term_length'] / np.maximum(
    df['description_length'], 1
)
df['title_to_desc_len_ratio'] = df['title_length'] / np.maximum(
    df['description_length'], 1
)
df['desc_to_title_len_ratio'] = df['description_length'] / np.maximum(
    df['title_length'], 1
)

# Word Count Ratios
df['search_to_title_word_ratio'] = df['search_term_word_count'] / np.maximum(
    df['title_word_count'], 1
)
df['search_to_desc_word_ratio'] = df['search_term_word_count'] / np.maximum(
    df['description_word_count'], 1
)
df['title_to_desc_word_ratio'] = df['title_word_count'] / np.maximum(
    df['description_word_count'], 1
)
df['combined_text'] = df['product_title'] + ' ' + df['product_description'] + ' ' + df['attr_text']

# Digit Density Ratio in Combined Text (Count of numeric digits / Len of combined_text)
df['combined_digit_count'] = df['combined_text'].str.count(r'\d')
df['combined_digit_ratio'] = df['combined_digit_count'] / np.maximum(
    df['combined_text'].str.len(), 1
)

In [ ]:
import re
import string
import pandas as pd

# Define punctuation set excluding hyphens '-'
PUNCT_EXCEPT_HYPHEN = "".join([c for c in string.punctuation if c != "-"])

# Custom stopword set
stopwords_vector = {
    "a",
    "an",
    "the",
    "in",
    "on",
    "of",
    "and",
    "is",
    "are",
    "for",
    "to",
    "it",
    "this",
    "that",
    "with",
    "as",
    "by",
    "at",
    "your",
    "you",
    "or",
    "be",
    "its",
    "has",
    "x",
    "from",
}


def clean_text_pipeline(
    text_series: pd.Series, stopwords_set: set[str]
) -> pd.Series:
  """Cleans whitespace, removes punctuation (except hyphens), lowercases,

  filters stopwords on split words, and returns a reconstructed string.
  """
  # 1. Clean whitespace and remove punctuation
  cleaned = (
      text_series.astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.translate(str.maketrans("", "", PUNCT_EXCEPT_HYPHEN))
      .str.lower()
  )

  # 2. Split into words, filter stopwords, and join back into a single string
  return cleaned.apply(
      lambda text: " ".join(
          [word for word in text.split() if word not in stopwords_set]
      )
  )


# Execute directly onto DataFrame
df["product_description"] = clean_text_pipeline(
    df["product_description"], stopwords_vector
)
df["search_term"] = clean_text_pipeline(df["search_term"], stopwords_vector)

In [ ]:
# ---------------------------------------------------------
# 2a. Average TF-IDF Score of Search Terms in Combined Text
# ---------------------------------------------------------
# Fit TF-IDF on the combined catalog text
combined_vectorizer = TfidfVectorizer(max_features=5000)
combined_tfidf = combined_vectorizer.fit_transform(df['product_title'])

# Map search terms to the vocabulary of the combined text matrix
count_vectorizer = CountVectorizer(vocabulary=combined_vectorizer.vocabulary_)
search_counts = count_vectorizer.transform(df['search_term'])
search_mask = (search_counts > 0).astype(float)

# Extract TF-IDF weights in the combined text for words present in the search term
matching_tfidf = combined_tfidf.multiply(search_mask)
sum_matching_tfidf = np.array(matching_tfidf.sum(axis=1)).ravel()

# Calculate average TF-IDF weight per search term token
df['search_tfidf_score_average_title'] = sum_matching_tfidf / np.maximum(
    df['search_term_word_count'], 1
)

In [ ]:
# ---------------------------------------------------------
# 2b. Average TF-IDF Score of Search Terms in Combined Text
# ---------------------------------------------------------
# Fit TF-IDF on the combined catalog text
combined_vectorizer = TfidfVectorizer(max_features=5000)
combined_tfidf = combined_vectorizer.fit_transform(df['product_description'])

# Map search terms to the vocabulary of the combined text matrix
count_vectorizer = CountVectorizer(vocabulary=combined_vectorizer.vocabulary_)
search_counts = count_vectorizer.transform(df['search_term'])
search_mask = (search_counts > 0).astype(float)

# Extract TF-IDF weights in the combined text for words present in the search term
matching_tfidf = combined_tfidf.multiply(search_mask)
sum_matching_tfidf = np.array(matching_tfidf.sum(axis=1)).ravel()

# Calculate average TF-IDF weight per search term token
df['search_tfidf_score_average_desc'] = sum_matching_tfidf / np.maximum(
    df['search_term_word_count'], 1
)

In [ ]:
# ---------------------------------------------------------
# 2c. Average TF-IDF Score of Search Terms in Combined Text
# ---------------------------------------------------------
# Fit TF-IDF on the combined catalog text
combined_vectorizer = TfidfVectorizer(max_features=5000)
combined_tfidf = combined_vectorizer.fit_transform(df['attr_text'])

# Map search terms to the vocabulary of the combined text matrix
count_vectorizer = CountVectorizer(vocabulary=combined_vectorizer.vocabulary_)
search_counts = count_vectorizer.transform(df['search_term'])
search_mask = (search_counts > 0).astype(float)

# Extract TF-IDF weights in the combined text for words present in the search term
matching_tfidf = combined_tfidf.multiply(search_mask)
sum_matching_tfidf = np.array(matching_tfidf.sum(axis=1)).ravel()

# Calculate average TF-IDF weight per search term token
df['search_tfidf_score_average_attr'] = sum_matching_tfidf / np.maximum(
    df['search_term_word_count'], 1
)

In [ ]:
# ---------------------------------------------------------
# 3. Rowwise Cosine Similarity Across All Three Field Pairs
# ---------------------------------------------------------
# Shared feature space across all three text sources
corpus = pd.concat(
    [df['product_title'], df['product_description'], df['search_term'], df['attr_text']]
)
joint_vectorizer = TfidfVectorizer(max_features=5000, ngram_range = (1, 3)).fit(corpus)

title_matrix = joint_vectorizer.transform(df['product_title'])
desc_matrix = joint_vectorizer.transform(df['product_description'])
search_matrix = joint_vectorizer.transform(df['search_term'])


# Vectorized helper function for row-wise cosine similarity computation
def rowwise_cosine_similarity(mat1, mat2):
  dot = np.array(mat1.multiply(mat2).sum(axis=1)).ravel()
  norm1 = np.sqrt(np.array(mat1.multiply(mat1).sum(axis=1)).ravel())
  norm2 = np.sqrt(np.array(mat2.multiply(mat2).sum(axis=1)).ravel())
  denominator = norm1 * norm2
  denominator[denominator == 0] = 1.0
  return dot / denominator


# Compute similarities for all three combinations
df['search_title_cosine_similarity'] = rowwise_cosine_similarity(
    search_matrix, title_matrix
)
df['description_search_cosine_similarity'] = rowwise_cosine_similarity(
    search_matrix, desc_matrix
)
df['title_description_cosine_similarity'] = rowwise_cosine_similarity(
    title_matrix, desc_matrix
)

# Ensure VADER lexicon is downloaded
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Load spaCy model for NER (requires: python -m spacy download en_core_web_sm)
nlp = spacy.load("en_core_web_sm")

In [ ]:
# ---------------------------------------------------------
# 4. VADER Sentiment Analysis
# ---------------------------------------------------------
# Calculate the compound sentiment score for the product description
# VADER combines a lexicon with rules for negation, punctuation, and intensifiers
df['desc_vader_compound'] = df['product_description'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

# Optional: Calculate for the search term to gauge user intent sentiment
df['search_vader_compound'] = df['search_term'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

To classify the Home Depot product catalog into 25 distinct departments with end-to-end auditability, we developed a glass-box hybrid classification engine that evolved across ten iterative optimization cycles. Beginning with a baseline dictionary model that left over 65% of the catalog unclassified in the "Other" bucket, we systematically constructed a three-pass regex engine targeting product titles, brand names, and explicit attribute metadata. To eliminate cross-department keyword collisions, we implemented a priority-weighted disambiguation hierarchy and a tiered execution cascade (locking in high-precision title and brand matches before evaluating broad attribute strings), successfully driving ambiguous classifications down to 0.00%. When rule-based dictionary expansion reached diminishing returns at 73.5% clean coverage, we leveraged the rule-labeled records as high-confidence ground truth to train a supervised Logistic Regression fallback model. By replacing absolute probability cutoffs with a relative margin-based thresholding scheme ($\Delta P \ge 0.15$), the engine safely classified specification-heavy items, elevating total catalog coverage to 84.44%. The complete architecture was ultimately packaged into a modular, production-ready `HomeDepotClassifier` class featuring automated source lineage logging, missing-column safety checks, and strict audit isolation for the remaining 15.56% unclassified pool.


In [ ]:
class HomeDepotClassifier:
  """Production Glass-Box Deterministic & ML Hybrid Classifier for Home Depot

  Catalog.

  Architecture:
  - Pass 1: High-precision Title + Brand exact noun regex matching.
  - Pass 2: Fallback to explicit attribute fields (e.g., 'attr_product_type').
  - Pass 3: Full attribute text regex matching.
  - Conflict Resolution: Deterministic priority hierarchy for multi-matches
  (0.00% ambiguity).
  - Pass 4: Calibrated Logistic Regression fallback with margin-based thresholding
            (P_top1 - P_top2 >= margin_threshold).
  """

  EXPANDED_PATTERNS = {
      'Appliances': [
          'refrigerator',
          'dishwasher',
          'microwave',
          'washing machine',
          'dryer',
          'freezer',
          'cooktop',
          'range hood',
          'beverage cooler',
          'ice maker',
          'compact fridge',
          'mini fridge',
          'range',
          'wall oven',
          'built in oven',
      ],
      'Automotive': [
          'car battery',
          'motor oil',
          'tire',
          'jack stand',
          'automotive',
          'wiper blade',
          'car care',
          'towing',
      ],
      'Bath & Faucets': [
          'faucet',
          'bathtub',
          'showerhead',
          'vanity',
          'toilet',
          'bath accessory',
          'shower door',
          'towel bar',
          'bath vanity',
          'shower base',
          'shower pan',
          'towel ring',
          'toilet paper holder',
          'bathroom sink',
          'lavatory',
          'medicine cabinet',
          'american standard',
          'kohler',
          'moen',
          'delta faucet',
          'bath sink',
      ],
      'Blinds & Window Treatments': [
          'window blind',
          'roller shade',
          'curtain rod',
          'plantation shutter',
          'vertical blind',
          'cellular shade',
          'drape',
          'window shade',
          'window film',
          'window tint',
      ],
      'Building Materials': [
          'plywood',
          'drywall',
          'concrete',
          'insulation',
          'roofing',
          'rebar',
          'siding',
          'cement',
          'lumber',
          'stud',
          'sheathing',
          'underlayment board',
          'pressure treated',
          'common actual',
          'furring strip',
          'oriented strand board',
          'osb board',
          'asphalt shingle',
          'fence panel',
          'chain link fence',
          'builders edge',
          'galvanized steel',
          'composite decking',
          'post cap',
          'deck post cap',
          'fence post cap',
          'building products',
          'wilsonart laminate',
          'laminate sheet',
          'countertop sheet',
          'amerimax home products',
          'amerimax',
          'gutter guard',
      ],
      'Cleaning': [
          'vacuum',
          'trash bag',
          'mop',
          'bleach',
          'disinfectant',
          'paper towel',
          'pressure washer detergent',
          'broom',
          'trash can',
          'cleaning wipes',
          'pressure washer',
          'power washer',
      ],
      'Decor & Furniture': [
          'wall art',
          'mirror',
          'area rug',
          'throw pillow',
          'accent chair',
          'desk',
          'headboard',
          'curtain',
          'home decorators collection',
          'martha stewart living',
          'accent table',
          'wall clock',
          'trademark fine art',
          'fine art',
          'wall tapestry',
          'framed canvas',
          'canvas wall art',
          'south shore',
          'south shore furniture',
          'shore furniture',
      ],
      'Doors & Windows': [
          'entry door',
          'interior door',
          'garage door',
          'window frame',
          'screen door',
          'skylight',
          'storm door',
          'front door',
          'prehung door',
          'slab door',
          'patio door',
          'bifold door',
          'window screen',
          'sliding door',
          'vinyl window',
          'fold door',
          'folding door',
          'shutters pair',
          'exterior shutter',
          'glass block window',
          'glass block',
      ],
      'Electrical': [
          'circuit breaker',
          'outlet',
          'extension cord',
          'junction box',
          'conduit',
          'electrical wire',
          'switch',
          'wire nut',
          'wall plate',
          'switch plate',
          'receptacle',
          'dimmer',
          'electrical box',
          'voltage tester',
          'commercial electric',
          'klein tools',
          'carbon monoxide alarm',
          'smoke carbon monoxide',
          'co alarm',
          'smoke detector',
      ],
      'Home Electronics': [
          'tv mount',
          'hdmi cable',
          'speaker',
          'security camera',
          'surge protector',
          'intercom',
          'ethernet cable',
          'tv wall mount',
      ],
      'Flooring, Tile, & Rugs': [
          'vinyl plank',
          'laminate flooring',
          'hardwood flooring',
          'grout',
          'ceramic tile',
          'porcelain tile',
          'underlayment',
          'wall tile',
          'floor tile',
          'mosaic tile',
          'backsplash tile',
          'tile case',
          'carpet tile',
          'transition strip',
          'stair nose',
          'door mat',
          'welcome mat',
          'entry mat',
          'stair tread',
          'wall cove base',
          'cove base',
      ],
      'Garage': [
          'garage door opener',
          'garage shelving',
          'workbenches',
          'parking assist',
          'garage flooring',
      ],
      'Hardware': [
          'deadbolt',
          'padlock',
          'drywall anchor',
          'hinge',
          'bracket',
          'screw',
          'nail',
          'bolt',
          'cabinet knob',
          'prime line',
          'corner brace',
          'fastener',
          'shelf bracket',
          'simpson strong',
          'strong tie',
          'post base',
          'joist hanger',
          'anchor bolt',
          'machine screw',
          'wood screw',
          'hillman group',
          'hillman',
          'fastener pack',
          'cabinet hardware',
      ],
      'Heating, Cooling, & Air Quality': [
          'air filter',
          'furnace filter',
          'air conditioner',
          'space heater',
          'thermostat',
          'dehumidifier',
          'duct',
          'vent cap',
          'return air grille',
          'pleasant hearth',
          'fireplace screen',
          'log rack',
          'firebox',
          'stove pipe',
          'chimney stove',
          'evaporative cooler',
          'cap shelf mantel',
          'fireplace mantel',
      ],
      'Holiday Decorations': [
          'christmas tree',
          'string lights',
          'halloween decoration',
          'wreath',
          'inflatable',
          'holiday projector',
          'home accents holiday',
      ],
      'Kitchen & Kitchenware': [
          'kitchen sink',
          'cabinet organizer',
          'backsplash',
          'cookware',
          'cutting board',
          'dish rack',
          'base cabinet',
          'wall cabinet',
          'kitchen cabinet',
          'pantry cabinet',
          'quartz countertop sample',
          'silestone',
          'countertop sample',
      ],
      'Lawn & Garden': [
          'lawn mower',
          'string trimmer',
          'leaf blower',
          'garden hose',
          'fertilizer',
          'mulch',
          'potting soil',
          'sprinkler',
          'weed eater',
          'grass seed',
          'landscape fabric',
          'john deere',
          'bed bug',
          'garden flag',
          'briggs amp stratton',
          'briggs & stratton',
          'synthetic lawn turf',
          'artificial turf',
      ],
      'Lighting & Ceiling Fans': [
          'ceiling fan',
          'chandelier',
          'light bulb',
          'vanity light',
          'sconce',
          'pendant light',
          'recessed light',
          'hampton bay',
          'lithonia lighting',
          'led light',
          'flush mount light',
          'track lighting',
          'landscape lighting',
          'solar powered light',
          'progress lighting',
          'collection light',
          'filament design',
          'floor lamp',
          'fan essentials',
          'torchieres',
      ],
      'Moulding & Millwork': [
          'baseboard',
          'crown moulding',
          'casing',
          'chair rail',
          'stair riser',
          'wainscoting',
          'moulding',
          'trim moulding',
          'shoe moulding',
          'ekena millwork',
          'ekena',
          'millwork bracket',
      ],
      'Outdoors': [
          'patio furniture',
          'charcoal grill',
          'fire pit',
          'gazebo',
          'patio umbrella',
          'pergola',
          'outdoor heater',
          'gas grill',
          'storage shed',
          'above ground pool',
          'solar pool blanket',
          'patio living concepts',
          'pool cover',
      ],
      'Paint': [
          'spray paint',
          'primer',
          'paintbrush',
          'paint roller',
          'interior paint',
          'exterior paint',
          'wood stain',
          'caulk',
          'deck paint',
          'sealant',
          'behr premium',
          'behr process',
          'rust oleum',
          'premium plus',
          'deckover',
      ],
      'Plumbing': [
          'sump pump',
          'water heater',
          'pvc pipe',
          'pipe fitting',
          'shut off valve',
          'drain auger',
          'pex tubing',
          'p trap',
          'ball valve',
          'copper pipe',
          'toilet tank',
          'sharkbite',
          'brass fitting',
          'charlotte pipe',
      ],
      'Smart Home': [
          'smart thermostat',
          'smart lock',
          'smart plug',
          'video doorbell',
          'smart light switch',
          'smart hub',
      ],
      'Storage & Organization': [
          'storage tote',
          'shelving unit',
          'moving box',
          'closet system',
          'storage bin',
          'deck box',
          'plastic storage',
          'honey can',
          'honey can do',
          'honey-can-do',
          'shelf liner',
          'con tact',
          'storage cabinet',
          'multipurpose shelf liner',
      ],
      'Tools': [
          'drill',
          'circular saw',
          'air compressor',
          'socket set',
          'ladder',
          'impact driver',
          'hammer',
          'tape measure',
          'lithium ion',
          'cordless drill',
          'batteries charger',
          'tool battery',
          'combo kit',
          'dewalt',
          'milwaukee',
          'ryobi',
          'saw blade',
          'reciprocating saw',
          'black decker',
          'black & decker',
          'tool box',
          'truck tool box',
          'sanding disc',
          'utility box',
      ],
  }

  PRIORITY_HIERARCHY = [
      'Tools',
      'Appliances',
      'Bath & Faucets',
      'Lighting & Ceiling Fans',
      'Doors & Windows',
      'Flooring, Tile, & Rugs',
      'Electrical',
      'Plumbing',
      'Paint',
      'Heating, Cooling, & Air Quality',
      'Kitchen & Kitchenware',
      'Blinds & Window Treatments',
      'Building Materials',
      'Hardware',
      'Cleaning',
      'Lawn & Garden',
      'Moulding & Millwork',
      'Storage & Organization',
      'Decor & Furniture',
      'Outdoors',
      'Holiday Decorations',
      'Home Electronics',
      'Automotive',
      'Smart Home',
      'Garage',
  ]

  def __init__(
      self,
      min_confidence: float = 0.35,
      min_margin: float = 0.15,
      max_features: int = 25000,
  ):
    self.min_confidence = min_confidence
    self.min_margin = min_margin
    self.max_features = max_features

    self.compiled_patterns = {
        dept: re.compile(
            r'\b(?:' + '|'.join(map(re.escape, keywords)) + r')\b',
            flags=re.IGNORECASE,
        )
        for dept, keywords in self.EXPANDED_PATTERNS.items()
    }

    self.vectorizer = TfidfVectorizer(
        max_features=self.max_features,
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words='english',
    )
    self.model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

  def _safe_series(self, df: pd.DataFrame, col_name: str) -> pd.Series:
    if col_name in df.columns:
      return df[col_name].fillna('').astype(str)
    return pd.Series('', index=df.index)

  def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
    """Executes the full 4-pass hybrid pipeline and returns the annotated

    DataFrame.
    """
    df = df.copy()

    # --- PASS 1 to 3: DETERMINISTIC RULE ENGINE ---
    title_s = self._safe_series(df, 'product_title')
    brand_s = self._safe_series(df, 'brand_name')
    attr_type_s = self._safe_series(df, 'attr_product_type')
    attr_text_s = self._safe_series(df, 'attr_text')

    df['department'] = 'Other'

    # Pass 1: Title + Brand
    pass1_text = (title_s + ' ' + brand_s).str.strip()
    p1_mat = pd.DataFrame(0, index=df.index, columns=self.compiled_patterns.keys())
    for dept, pat in self.compiled_patterns.items():
      p1_mat[dept] = pass1_text.str.contains(pat, regex=True).astype(int)

    p1_sum = p1_mat.sum(axis=1)
    df.loc[p1_sum == 1, 'department'] = p1_mat[p1_sum == 1].idxmax(axis=1)

    multi_p1 = p1_sum > 1
    if multi_p1.any():
      p1_multi_df = p1_mat[multi_p1]
      for dept in self.PRIORITY_HIERARCHY:
        hits = p1_multi_df[dept] == 1
        if hits.any():
          unassigned = hits & (df.loc[p1_multi_df.index, 'department'] == 'Other')
          if unassigned.any():
            df.loc[unassigned[unassigned].index, 'department'] = dept

    # Pass 2: Explicit Attribute Type (Fallback)
    p2_mask = (df['department'] == 'Other') & (attr_type_s != '')
    if p2_mask.any():
      p2_text = attr_type_s[p2_mask]
      p2_mat = pd.DataFrame(
          0, index=df[p2_mask].index, columns=self.compiled_patterns.keys()
      )
      for dept, pat in self.compiled_patterns.items():
        p2_mat[dept] = p2_text.str.contains(pat, regex=True).astype(int)

      p2_sum = p2_mat.sum(axis=1)
      df.loc[p2_sum[p2_sum == 1].index, 'department'] = p2_mat.loc[
          p2_sum == 1
      ].idxmax(axis=1)

      multi_p2 = p2_sum > 1
      if multi_p2.any():
        p2_multi_df = p2_mat[multi_p2]
        for dept in self.PRIORITY_HIERARCHY:
          hits = p2_multi_df[dept] == 1
          if hits.any():
            unassigned = hits & (
                df.loc[p2_multi_df.index, 'department'] == 'Other'
            )
            if unassigned.any():
              df.loc[unassigned[unassigned].index, 'department'] = dept

    # Pass 3: Full Attribute Text (Fallback)
    p3_mask = df['department'] == 'Other'
    if p3_mask.any():
      p3_text = attr_text_s[p3_mask]
      p3_mat = pd.DataFrame(
          0, index=df[p3_mask].index, columns=self.compiled_patterns.keys()
      )
      for dept, pat in self.compiled_patterns.items():
        p3_mat[dept] = p3_text.str.contains(pat, regex=True).astype(int)

      p3_sum = p3_mat.sum(axis=1)
      df.loc[p3_sum[p3_sum == 1].index, 'department'] = p3_mat.loc[
          p3_sum == 1
      ].idxmax(axis=1)

      multi_p3 = p3_sum > 1
      if multi_p3.any():
        p3_multi_df = p3_mat[multi_p3]
        for dept in self.PRIORITY_HIERARCHY:
          hits = p3_multi_df[dept] == 1
          if hits.any():
            unassigned = hits & (
                df.loc[p3_multi_df.index, 'department'] == 'Other'
            )
            if unassigned.any():
              df.loc[unassigned[unassigned].index, 'department'] = dept

    df['classification_source'] = np.where(
        df['department'] != 'Other', 'Rule Engine', 'Unclassified'
    )

    # --- PASS 4: CALIBRATED ML FALLBACK ---
    train_mask = df['department'] != 'Other'
    predict_mask = df['department'] == 'Other'

    if not predict_mask.any():
      return df

    search_s = self._safe_series(df, 'search_term')
    text_corpus = (
        title_s + ' ' + search_s + ' ' + brand_s + ' ' + attr_text_s
    ).str.strip()

    X_train = self.vectorizer.fit_transform(text_corpus[train_mask])
    y_train = df.loc[train_mask, 'department']
    X_predict = self.vectorizer.transform(text_corpus[predict_mask])

    # Filter classes with <3 ground truth samples
    class_counts = y_train.value_counts()
    valid_classes = class_counts[class_counts >= 3].index
    valid_indices = y_train.isin(valid_classes)

    self.model.fit(X_train[valid_indices.values], y_train[valid_indices])

    probs = self.model.predict_proba(X_predict)
    sorted_probs = np.sort(probs, axis=1)

    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2]
    margins = top1 - top2
    preds = self.model.classes_[np.argmax(probs, axis=1)]

    accepted = (top1 >= self.min_confidence) & (margins >= self.min_margin)
    predict_idx = df[predict_mask].index
    assigned_idx = predict_idx[accepted]

    df.loc[assigned_idx, 'department'] = preds[accepted]
    df.loc[assigned_idx, 'classification_source'] = (
        f'ML Fallback (Margin >= {self.min_margin})'
    )

    df['ml_confidence'] = 0.0
    df['ml_margin'] = 0.0
    df.loc[predict_idx, 'ml_confidence'] = top1
    df.loc[predict_idx, 'ml_margin'] = margins

    return df

  @staticmethod
  def print_summary_report(df: pd.DataFrame):
    """Outputs structured catalog coverage and classification source audit

    metrics.
    """
    print('=== CATALOG DEPARTMENT BREAKDOWN ===')
    print(
        df['department']
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_string()
    )

    print('\n=== AUDIT SOURCE DISTRIBUTION ===')
    print(df['classification_source'].value_counts().to_string())

    classified_pct = (df['department'] != 'Other').mean() * 100
    print(
        f'\nTotal Classified Coverage: {classified_pct:.2f}% (Dropped:'
        f' {100 - classified_pct:.2f}%)'
    )

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# ---------------------------------------------------------
# 1. RUN HYBRID CLASSIFIER & OVERWRITE `df` IN-PLACE
# ---------------------------------------------------------
# Assigning directly back to `df` fixes the KeyError: 'department'
pipeline = HomeDepotClassifier(min_confidence=0.35, min_margin=0.15)

df = pipeline.fit_transform(df)

# Print execution audit report
pipeline.print_summary_report(df)

# ---------------------------------------------------------
# 2. DEFINE CONTINUOUS FEATURES (INCLUDING FLIPPED RATIOS)
# ---------------------------------------------------------
CONTINUOUS_FEATURES = [
    'title_length',
    'description_length',
    'search_term_length',
    'attr_length',
    'title_word_count',
    'description_word_count',
    'search_term_word_count',
    'title_noise_count',
    'title_noise_ratio',
    'search_to_title_len_ratio',
    'search_to_desc_len_ratio',
    'title_to_desc_len_ratio',
    'search_to_title_word_ratio',
    'search_to_desc_word_ratio',
    'title_to_desc_word_ratio',
    'combined_digit_count',
    'combined_digit_ratio',
    'search_tfidf_score_average_title',
    'search_tfidf_score_average_desc',
    'search_tfidf_score_average_attr',
    'search_title_cosine_similarity',
    'description_search_cosine_similarity',
    'title_description_cosine_similarity',
    'desc_vader_compound',
    'search_vader_compound',
]

# ---------------------------------------------------------
# 3. SPEARMAN ANALYSIS FUNCTION
# ---------------------------------------------------------
def analyze_department_spearman(
    df: pd.DataFrame,
    split_val: float = 2.33,
    use_department_median: bool = False,
    min_sample_size: int = 30,
) -> pd.DataFrame:
  """Executes Spearman rank correlation and High vs Low group mean comparisons

  across departments.
  """
  df_work = df.copy()
  valid_features = [f for f in CONTINUOUS_FEATURES if f in df_work.columns]
  results = []

  for dept, dept_df in df_work.groupby('department'):
    if len(dept_df) < min_sample_size:
      continue

    # Determine cutoff threshold (Global 2.33 vs Department Median)
    cutoff = (
        dept_df['relevance'].median() if use_department_median else split_val
    )

    high_mask = dept_df['relevance'] > cutoff
    low_mask = dept_df['relevance'] <= cutoff

    if high_mask.sum() == 0 or low_mask.sum() == 0:
      continue

    for feature in valid_features:
      high_mean = dept_df.loc[high_mask, feature].mean()
      low_mean = dept_df.loc[low_mask, feature].mean()
      delta = high_mean - low_mean

      # Spearman Rank Correlation
      rho, p_val = spearmanr(dept_df[feature], dept_df['relevance'])

      results.append({
          'department': dept,
          'feature': feature,
          'cutoff_used': round(cutoff, 2),
          'high_mean (> cutoff)': round(high_mean, 4),
          'low_mean (<= cutoff)': round(low_mean, 4),
          'delta (high - low)': round(delta, 4),
          'spearman_rho': round(rho, 4),
          'p_value': round(p_val, 5),
          'significant': p_val < 0.05,
      })

  return pd.DataFrame(results)


# ---------------------------------------------------------
# 4. EXECUTE SPEARMAN ANALYSIS & GENERATE SUMMARY
# ---------------------------------------------------------
spearman_results = analyze_department_spearman(
    df, split_val=2.33, use_department_median=False
)

summary_list = []
for dept, dept_res in spearman_results.groupby('department'):
  sig_res = dept_res[dept_res['significant']].copy()
  if sig_res.empty:
    continue

  top_pos = sig_res.sort_values(by='spearman_rho', ascending=False).head(3)
  top_neg = sig_res.sort_values(by='spearman_rho', ascending=True).head(3)

  pos_str = ', '.join(
      [f"{r['feature']} ({r['spearman_rho']:+.3f})" for _, r in top_pos.iterrows()]
  )
  neg_str = ', '.join(
      [f"{r['feature']} ({r['spearman_rho']:+.3f})" for _, r in top_neg.iterrows()]
  )

  summary_list.append({
      'Department': dept,
      'Top Positive Drivers': pos_str if pos_str else 'None',
      'Top Negative Drivers': neg_str if neg_str else 'None',
  })

summary_df = pd.DataFrame(summary_list)

print('\n=== SPEARMAN DEPARTMENT RELEVANCE DRIVERS ===')
print(summary_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Pivot spearman_results into a Department x Feature correlation matrix
heatmap_matrix = spearman_results.pivot(
    index='department', columns='feature', values='spearman_rho'
)

# 2. Sort both Departments (rows) and Features (columns) alphabetically
heatmap_matrix = heatmap_matrix.sort_index(ascending=True)
heatmap_matrix = heatmap_matrix.reindex(
    sorted(heatmap_matrix.columns), axis=1
)

# 3. Generate Heatmap
plt.figure(figsize=(22, 12), dpi=150)

# Create diverging heatmap centered at 0
ax = sns.heatmap(
    heatmap_matrix,
    cmap='vlag',  # Red (positive) / Blue (negative) / White (neutral)
    center=0,
    vmin=-0.4,
    vmax=0.4,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    linecolor='#e0e0e0',
    cbar_kws={'label': 'Spearman Correlation (ρ)', 'shrink': 0.8},
    annot_kws={'size': 8},
)

# 4. Professional Formatting
plt.title(
    'Spearman Correlation Matrix: Continuous Features vs. Search Relevance'
    ' (Alphabetical)',
    fontsize=16,
    pad=20,
    weight='bold',
)
plt.xlabel('Continuous Metrics', fontsize=12, labelpad=12, weight='bold')
plt.ylabel('Department', fontsize=12, labelpad=12, weight='bold')

plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9.5)
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------
# 1. Compute Department Relevance Summary
# ---------------------------------------------------------
dept_summary = (
    df.groupby('department')
    .agg(
        n_pairs=('relevance', 'count'),
        n_queries=('search_term', 'nunique'),
        n_titles=('product_title', 'nunique'),
        mean_relevance=('relevance', 'mean'),
        median_relevance=('relevance', 'median'),
        std_relevance=('relevance', 'std'),
        low_relevance_pct=('relevance', lambda x: (x < 2.0).mean() * 100),
        high_relevance_pct=('relevance', lambda x: (x >= 2.5).mean() * 100),
    )
    .sort_values('low_relevance_pct', ascending=False)
)

print('=== UPDATED DEPARTMENT SUMMARY ===')
display(dept_summary.round(3))

# ---------------------------------------------------------
# 2. Bar Chart Ranked by Percentage of Low-Relevance Matches
# ---------------------------------------------------------
plt.figure(figsize=(12, 8), dpi=150)

# Sort ascending so highest low-relevance % appears at the top of the horizontal bar plot
plot_data = dept_summary.sort_values('low_relevance_pct', ascending=True)

plt.barh(
    plot_data.index,
    plot_data['low_relevance_pct'],
    color='#d9534f',
    edgecolor='#333333',
    height=0.65,
)

plt.xlabel(
    'Search–Product Pairs with Relevance < 2.0 (%)',
    fontsize=11,
    weight='bold',
    labelpad=10,
)
plt.ylabel('Department', fontsize=11, weight='bold', labelpad=10)
plt.title(
    'Low-Relevance Search Rate (< 2.0) by Department',
    fontsize=14,
    pad=15,
    weight='bold',
)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
df[df.department == 'Heating, Cooling, & Air Quality'].product_description.sample().iloc[0]

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def diagnose_description_content(df: pd.DataFrame, min_sample_size: int = 30):
    df_work = df.copy()

    # 1. Feature Engineering on Description Content Structure
    # Spec/Measurement Density (matches numbers followed by units: 12 in, 20v, 5 gal, etc.)
    spec_pattern = re.compile(r'\b\d+(?:\.\d+)?\s*(?:in|ft|gal|v|volt|amp|lb|lbs|oz|mm|cm|m|w|watt|hp|qt|sq)\b|\b\d+\b', re.IGNORECASE)
    df_work['desc_spec_count'] = df_work['product_description'].apply(lambda x: len(spec_pattern.findall(str(x))))
    df_work['desc_spec_density'] = df_work['desc_spec_count'] / np.maximum(df_work['description_word_count'], 1)

    # Check for Bullet/List Structure
    df_work['has_bullet_structure'] = df_work['product_description'].str.contains(r'bullet|\*|\-|•|\d\.', case=False, regex=True).astype(int)

    results = []

    print("=== DESCRIPTION CONTENT DIAGNOSTIC REPORT ===")

    for dept, dept_df in df_work.groupby('department'):
        if len(dept_df) < min_sample_size or dept == 'Other':
            continue

        high_mask = dept_df['relevance'] > 2.33
        low_mask = dept_df['relevance'] <= 2.33

        if high_mask.sum() < 5 or low_mask.sum() < 5:
            continue

        # Group Means
        high_len = dept_df.loc[high_mask, 'description_length'].mean()
        low_len = dept_df.loc[low_mask, 'description_length'].mean()

        high_words = dept_df.loc[high_mask, 'description_word_count'].mean()
        low_words = dept_df.loc[low_mask, 'description_word_count'].mean()

        high_specs = dept_df.loc[high_mask, 'desc_spec_count'].mean()
        low_specs = dept_df.loc[low_mask, 'desc_spec_count'].mean()

        high_cos = dept_df.loc[high_mask, 'description_search_cosine_similarity'].mean()
        low_cos = dept_df.loc[low_mask, 'description_search_cosine_similarity'].mean()

        results.append({
            'Department': dept,
            'High_Len_Chars': round(high_len, 1),
            'Low_Len_Chars': round(low_len, 1),
            'Delta_Chars': round(high_len - low_len, 1),
            'High_Word_Count': round(high_words, 1),
            'Low_Word_Count': round(low_words, 1),
            'High_Specs_Count': round(high_specs, 1),
            'Low_Specs_Count': round(low_specs, 1),
            'Delta_Specs': round(high_specs - low_specs, 1),
            'High_Search_Cosine': round(high_cos, 3),
            'Low_Search_Cosine': round(low_cos, 3),
        })

    diag_df = pd.DataFrame(results).sort_values(by='Delta_Chars', ascending=False)
    display(diag_df)

    # 2. Extract Top Distinctive Terms in Low vs High Descriptions (Global sample)
    print("\n=== TOP DISTINCTIVE N-GRAMS IN HIGH vs LOW RELEVANCE DESCRIPTIONS ===")
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))

    high_docs = df_work[df_work['relevance'] > 2.33]['product_description']
    low_docs = df_work[df_work['relevance'] <= 2.33]['product_description']

    X_high = vectorizer.fit_transform(high_docs)
    high_freq = np.asarray(X_high.mean(axis=0)).ravel()

    X_low = vectorizer.transform(low_docs)
    low_freq = np.asarray(X_low.mean(axis=0)).ravel()

    words = np.array(vectorizer.get_feature_names_out())
    diff = high_freq - low_freq

    top_high_terms = words[np.argsort(diff)[-15:]][::-1]
    top_low_terms = words[np.argsort(diff)[:15]]

    print("Top Terms Over-represented in HIGH Relevance Descriptions:")
    print(", ".join(top_high_terms))

    print("\nTop Terms Over-represented in LOW Relevance Descriptions:")
    print(", ".join(top_low_terms))

# Execute diagnostic
diagnose_description_content(df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def plot_micro_heatmap(df: pd.DataFrame, n_depts: int = 5, n_features: int = 5):
    # 1. Compute departmental Spearman correlation matrix
    results = []
    features = [c for c in df.columns if c not in ['relevance', 'department', 'product_uid', 'id', 'product_title', 'product_description', 'search_term']]

    for dept, dept_df in df.groupby('department'):
        if len(dept_df) < 30:
            continue
        low_rate = (dept_df['relevance'] <= 2.33).mean()
        row = {'department': dept, 'low_relevance_rate': low_rate}
        for f in features:
            if dept_df[f].nunique() > 1:
                rho, _ = spearmanr(dept_df[f], dept_df['relevance'])
                row[f] = rho
            else:
                row[f] = 0.0
        results.append(row)

    corr_df = pd.DataFrame(results).set_index('department')

    # 2. Select top priority departments (worst low-relevance rates)
    top_depts = corr_df.sort_values(by='low_relevance_rate', ascending=False).head(n_depts).index

    # 3. Select top features by average absolute correlation magnitude across catalog
    feature_cols = [c for c in corr_df.columns if c != 'low_relevance_rate']
    top_features = corr_df[feature_cols].abs().mean(axis=0).sort_values(ascending=False).head(n_features).index

    # 4. Filter matrix
    micro_matrix = corr_df.loc[top_depts, top_features]

    # 5. Plot
    plt.figure(figsize=(10, 5))
    sns.heatmap(
        micro_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-0.4,
        vmax=0.4,
        cbar_kws={'label': 'Spearman Correlation (ρ)'},
        linewidths=1,
        linecolor='white'
    )
    plt.title(f"Worst {n_depts} Departments by Relevance vs. High-Impact Relevance Drivers", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("Feature Levers", fontweight='bold')
    plt.ylabel("Department", fontweight='bold')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()

# Run Micro-Heatmap
plot_micro_heatmap(df, n_depts=6, n_features=5)

In [ ]:
def plot_content_gap_bars(df: pd.DataFrame):
    high_mask = df['relevance'] > 2.33
    low_mask = df['relevance'] <= 2.33

    eval_features = {
        'search_title_cosine_similarity': 'Search-Title Similarity',
        'search_tfidf_score_average_title': 'Title TF-IDF Alignment',
        'description_search_cosine_similarity': 'Search-Desc Similarity',
        'desc_spec_count': 'Technical Spec Count',
        'description_length': 'Description Length',
        'desc_vader_compound': 'Sentiment Score (Control)'
    }

    gaps = []
    for feat, label in eval_features.items():
        if feat in df.columns:
            low_val = df.loc[low_mask, feat].mean()
            high_val = df.loc[high_mask, feat].mean()
            pct_gap = ((high_val - low_val) / (abs(low_val) if low_val != 0 else 1)) * 100
            gaps.append({'Feature': label, 'Pct_Gap': pct_gap})

    gap_df = pd.DataFrame(gaps).sort_values(by='Pct_Gap')

    # Plot
    plt.figure(figsize=(9, 5))
    colors = ['#2b5c8f' if x > 0 else '#d95f02' for x in gap_df['Pct_Gap']]
    bars = plt.barh(gap_df['Feature'], gap_df['Pct_Gap'], color=colors, height=0.6)

    plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
    plt.title("Catalog Content Gap: High vs. Low Relevance Listings (% Delta)", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("% Gap (High Relevance vs Low Relevance Benchmark)", fontweight='bold')

    # Annotate bar values
    for bar in bars:
        width = bar.get_width()
        offset = 0.5 if width >= 0 else -3.5
        plt.text(width + offset, bar.get_y() + bar.get_height()/2, f"{width:+.1f}%",
                 va='center', fontsize=10, fontweight='bold')

    plt.xlim(min(gap_df['Pct_Gap']) - 8, max(gap_df['Pct_Gap']) + 8)
    plt.tight_layout()
    plt.show()

# Run Content Gap Chart
plot_content_gap_bars(df)

In [ ]:
def plot_headroom_dumbbell(df: pd.DataFrame, metric: str = 'search_title_cosine_similarity', metric_label: str = 'Search-Title Cosine Similarity'):
    dept_stats = []

    for dept, dept_df in df.groupby('department'):
        if len(dept_df) < 30 or dept == 'Other':
            continue
        low_val = dept_df.loc[dept_df['relevance'] <= 2.33, metric].mean()
        high_val = dept_df.loc[dept_df['relevance'] > 2.33, metric].mean()
        gap = high_val - low_val
        dept_stats.append({'department': dept, 'Low': low_val, 'High': high_val, 'Gap': gap})

    stats_df = pd.DataFrame(dept_stats).sort_values(by='Gap', ascending=True).tail(12) # Top 12 departments

    y_range = range(len(stats_df))
    plt.figure(figsize=(10, 6))

    # Draw connecting lines
    plt.hlines(y=y_range, xmin=stats_df['Low'], xmax=stats_df['High'], color='grey', alpha=0.6, linewidth=2)

    # Draw Low and High points
    plt.scatter(stats_df['Low'], y_range, color='#d95f02', alpha=1, s=80, label='Low Relevance Baseline (≤2.33)')
    plt.scatter(stats_df['High'], y_range, color='#2b5c8f', alpha=1, s=80, label='High Relevance Benchmark (>2.33)')

    plt.yticks(y_range, stats_df['department'], fontweight='bold')
    plt.xlabel(f"Average {metric_label}", fontweight='bold')
    plt.title(f"Optimization Headroom by Department: {metric_label}", fontsize=13, fontweight='bold', pad=15)
    plt.legend(loc='lower right', frameon=True)
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

# Run Dumbbell Chart
plot_headroom_dumbbell(df, metric='search_title_cosine_similarity', metric_label='Search-Title Cosine Similarity')

In [ ]:
def plot_global_driver_tornado(df: pd.DataFrame):
    features = [
        'search_tfidf_score_average_title',
        'search_title_cosine_similarity',
        'search_tfidf_score_average_desc',
        'description_search_cosine_similarity',
        'attr_length',
        'description_length',
        'desc_vader_compound',
        'title_noise_ratio',
        'desc_to_title_len_ratio',
        'title_length'
    ]

    readable_labels = {
        'search_tfidf_score_average_title': 'Title TF-IDF Score',
        'search_title_cosine_similarity': 'Search-Title Cosine Sim',
        'search_tfidf_score_average_desc': 'Desc TF-IDF Score',
        'description_search_cosine_similarity': 'Search-Desc Cosine Sim',
        'attr_length': 'Attribute Length',
        'description_length': 'Description Length',
        'desc_vader_compound': 'Desc Sentiment (VADER)',
        'title_noise_ratio': 'Title Noise Ratio',
        'desc_to_title_len_ratio': 'Desc-to-Title Ratio',
        'title_length': 'Title Character Length'
    }

    corrs = []
    for f in features:
        if f in df.columns:
            rho, _ = spearmanr(df[f], df['relevance'])
            corrs.append({'feature': readable_labels.get(f, f), 'rho': rho})

    corr_df = pd.DataFrame(corrs).sort_values(by='rho', ascending=True)

    plt.figure(figsize=(9, 6))
    colors = ['#d95f02' if r > 0.05 else ('#2b5c8f' if r < -0.05 else '#a6a6a6') for r in corr_df['rho']]

    bars = plt.barh(corr_df['feature'], corr_df['rho'], color=colors, height=0.6)
    plt.axvline(0, color='black', linewidth=1)

    plt.title("Global Search Relevance Drivers (Spearman Correlation ρ)", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("Correlation with Search Relevance (ρ)", fontweight='bold')
    plt.xlim(-0.35, 0.40)

    # Annotate values
    for bar in bars:
        val = bar.get_width()
        offset = 0.01 if val >= 0 else -0.04
        plt.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:+.2f}",
                 va='center', fontsize=10, fontweight='bold')

    plt.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

# Run Tornado Chart
plot_global_driver_tornado(df)

In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_noise_ratio_with_title_examples(df: pd.DataFrame):
    target_depts = ['Plumbing', 'Storage & Organization']
    comp_df = df[df['department'].isin(target_depts)].copy()

    if comp_df.empty:
        print("Target departments not found in DataFrame.")
        return

    # 1. Compute departmental mean noise ratios
    dept_stats = comp_df.groupby('department')['title_noise_ratio'].mean().reset_index()

    # 2. Extract Worst, Average, and Best product titles per department
    title_examples = {}
    for dept in target_depts:
        sub_df = comp_df[comp_df['department'] == dept].copy()
        mean_val = dept_stats.loc[dept_stats['department'] == dept, 'title_noise_ratio'].values[0]

        # Worst Title (Highest noise ratio)
        worst_row = sub_df.loc[sub_df['title_noise_ratio'].idxmax()]

        # Best Title (Lowest noise ratio)
        best_row = sub_df.loc[sub_df['title_noise_ratio'].idxmin()]

        # Average Title (Closest noise ratio to mean)
        sub_df['dist_from_mean'] = (sub_df['title_noise_ratio'] - mean_val).abs()
        avg_row = sub_df.loc[sub_df['dist_from_mean'].idxmin()]

        title_examples[dept] = {
            'Worst': (worst_row['product_title'], worst_row['title_noise_ratio']),
            'Average': (avg_row['product_title'], avg_row['title_noise_ratio']),
            'Best': (best_row['product_title'], best_row['title_noise_ratio'])
        }

    # 3. Render Single Bar Chart (Noise Ratio Only)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.set_style("whitegrid")
    palette = {'Plumbing': '#d95f02', 'Storage & Organization': '#2b5c8f'}

    bars = sns.barplot(
        x='department',
        y='title_noise_ratio',
        data=dept_stats,
        ax=ax,
        palette=palette,
        edgecolor='black',
        width=0.45
    )

    ax.set_title('Avg Title Noise Ratio (Lower is Better)', fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Department', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Ratio of Noise to Total Title', fontweight='bold', fontsize=11)

    max_ratio = dept_stats['title_noise_ratio'].max()
    ax.set_ylim(0, max_ratio * 1.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    # Annotate percentage values above bars
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f'{height:.1%}',
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom',
            xytext=(0, 6), textcoords='offset points',
            fontsize=12, fontweight='bold'
        )

    plt.suptitle('Operational Comparison: Title "Noise" Ratio Impact', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # 4. Print Executive Title Examples Report
    print("=" * 80)
    print("PRODUCT TITLE NOISE BREAKDOWN (Worst vs. Average vs. Best)")
    print("=" * 80)
    for dept, examples in title_examples.items():
        print(f"\n--- {dept.upper()} ---")
        print(f"• WORST TITLE (Ratio: {examples['Worst'][1]:.1%}):")
        print(f"  \"{examples['Worst'][0]}\"")
        print(f"• AVERAGE TITLE (Ratio: {examples['Average'][1]:.1%}):")
        print(f"  \"{examples['Average'][0]}\"")
        print(f"• BEST TITLE (Ratio: {examples['Best'][1]:.1%}):")
        print(f"  \"{examples['Best'][0]}\"")
    print("=" * 80)

# Run visualization & extract titles
plot_noise_ratio_with_title_examples(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_noise_ratio_overlapping_histogram(df: pd.DataFrame):
    target_depts = ['Plumbing', 'Storage & Organization']
    comp_df = df[df['department'].isin(target_depts)].copy()

    if comp_df.empty:
        print("Target departments not found in DataFrame.")
        return

    # Set aesthetic
    plt.figure(figsize=(10, 6))
    sns.set_style("white")

    colors = {'Plumbing': '#F96302', 'Storage & Organization': '#8C2A00'}

    # Plot overlapping histograms (Bars only)
    for dept in target_depts:
        sub_df = comp_df[comp_df['department'] == dept]

        # Overlapping semi-transparent histogram bars
        sns.histplot(
            sub_df['title_noise_ratio'],
            kde=False,
            stat="density",
            common_norm=False,
            color=colors[dept],
            label=dept,
            alpha=0.4,
            edgecolor="white",
            linewidth=0.5
        )


    plt.title('Title Noise Ratio Distribution Shift: Plumbing vs. Storage & Organization', fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Title Noise Ratio (% of Title Words Identified as Noise)', fontweight='bold', fontsize=11)
    plt.ylabel('Density', fontweight='bold', fontsize=11)

    # Format X-axis ticks as percentages
    plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

    plt.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=10)
    plt.grid(axis='y', linestyle=':', alpha=0.4)
    plt.tight_layout()
    plt.show()

# Run overlapping histogram plot
plot_noise_ratio_overlapping_histogram(df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def plot_micro_heatmap(df: pd.DataFrame, n_depts: int = 5, n_features: int = 5):
    # Business-friendly label dictionary to map variable names to presentation titles
    feature_label_map = {
        'search_tfidf_score_average_title': 'Title Keyword Precision (TF-IDF)',
        'search_title_cosine_similarity': 'Search-Title Semantic Match',
        'search_tfidf_score_average_desc': 'Description Keyword Precision',
        'description_search_cosine_similarity': 'Search-Desc Semantic Match',
        'desc_spec_count': 'Technical Spec Density',
        'description_length': 'Description Character Length',
        'description_word_count': 'Description Word Count',
        'title_noise_ratio': 'Title Noise Ratio',
        'title_noise_count': 'Title Noise Count',
        'title_length': 'Title Character Length',
        'desc_vader_compound': 'Marketing Tone / Sentiment',
        'attr_length': 'Attribute Volume',
        'desc_to_title_len_ratio': 'Desc-to-Title Length Ratio'
    }

    # 1. Compute departmental Spearman correlation matrix
    results = []
    features = [c for c in df.columns if c not in ['relevance', 'department', 'product_uid', 'id', 'product_title', 'product_description', 'search_term']]

    for dept, dept_df in df.groupby('department'):
        if len(dept_df) < 30:
            continue
        low_rate = (dept_df['relevance'] <= 2.33).mean()
        row = {'department': dept, 'low_relevance_rate': low_rate}
        for f in features:
            if dept_df[f].nunique() > 1:
                rho, _ = spearmanr(dept_df[f], dept_df['relevance'])
                row[f] = rho
            else:
                row[f] = 0.0
        results.append(row)

    corr_df = pd.DataFrame(results).set_index('department')

    # 2. Select top priority departments (worst low-relevance rates)
    top_depts = corr_df.sort_values(by='low_relevance_rate', ascending=False).head(n_depts).index

    # 3. Select top features by average absolute correlation magnitude across catalog
    feature_cols = [c for c in corr_df.columns if c != 'low_relevance_rate']
    top_features = corr_df[feature_cols].abs().mean(axis=0).sort_values(ascending=False).head(n_features).index

    # 4. Filter matrix and rename variable names to clean executive titles
    micro_matrix = corr_df.loc[top_depts, top_features].copy()

    # Apply business label map (with automatic fallback to Title Case formatting)
    micro_matrix.columns = [
        feature_label_map.get(col, col.replace('_', ' ').title())
        for col in micro_matrix.columns
    ]

    # 5. Plot heatmap
    plt.figure(figsize=(11, 5.5))
    sns.heatmap(
        micro_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-0.4,
        vmax=0.4,
        cbar_kws={'label': 'Spearman Correlation (ρ)'},
        linewidths=1,
        linecolor='white'
    )
    plt.title(f"Priority Matrix: Worst {n_depts} Departments vs. Top High-Impact Drivers", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("Content & Optimization Levers", fontweight='bold', labelpad=10)
    plt.ylabel("Priority Department", fontweight='bold')
    plt.xticks(rotation=25, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.tight_layout()
    plt.show()

# Run Micro-Heatmap
plot_micro_heatmap(df, n_depts=6, n_features=5)

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_relevance_high_low_split(df: pd.DataFrame):
    """
    Plots a High/Low Relevance split (threshold = 2.33) across 8 specific department-metric pairs:
      1-5. title_noise_ratio (Storage, Windows, Plumbing, Electronics, Heating/Cooling)
      6-7. desc_vader_compound (Appliances, Electronics)
      8.   description_length (Heating, Cooling, & Air Quality)
    """
    # 1. Define High/Low Relevance Tier
    df = df.copy()
    df['Relevance_Tier'] = np.where(df['relevance'] <= 2.33, 'Low Relevance (≤ 2.33)', 'High Relevance (> 2.33)')

    # 2. Define the 8 specific Department-Metric Target Pairs
    targets = [
        # Title Noise Ratio Group
        {'dept': 'Storage and Organization', 'metric': 'title_noise_ratio', 'group': 'Title Noise Ratio (%)', 'unit': 'pct'},
        {'dept': 'Windows and Blinds', 'metric': 'title_noise_ratio', 'group': 'Title Noise Ratio (%)', 'unit': 'pct'},
        {'dept': 'Plumbing', 'metric': 'title_noise_ratio', 'group': 'Title Noise Ratio (%)', 'unit': 'pct'},
        {'dept': 'Home Electronics', 'metric': 'title_noise_ratio', 'group': 'Title Noise Ratio (%)', 'unit': 'pct'},
        {'dept': 'Heating, Cooling, & Air Quality', 'metric': 'title_noise_ratio', 'group': 'Title Noise Ratio (%)', 'unit': 'pct'},

        # Sentiment Group
        {'dept': 'Appliances', 'metric': 'desc_vader_compound', 'group': 'Description Sentiment (VADER)', 'unit': 'score'},
        {'dept': 'Home Electronics', 'metric': 'desc_vader_compound', 'group': 'Description Sentiment (VADER)', 'unit': 'score'},

        # Description Length Group
        {'dept': 'Heating, Cooling, & Air Quality', 'metric': 'description_length', 'group': 'Description Length (Chars)', 'unit': 'num'},
    ]

    # 3. Aggregate calculated mean metrics
    records = []
    for t in targets:
        sub_df = df[df['department'] == t['dept']]
        if sub_df.empty or t['metric'] not in sub_df.columns:
            continue

        means = sub_df.groupby('Relevance_Tier')[t['metric']].mean().reset_index()
        for _, row in means.iterrows():
            records.append({
                'Department': t['dept'],
                'Metric': t['metric'],
                'Metric Group': t['group'],
                'Unit': t['unit'],
                'Relevance Tier': row['Relevance_Tier'],
                'Value': row[t['metric']]
            })

    plot_df = pd.DataFrame(records)
    if plot_df.empty:
        print("No matching department or metric data found in DataFrame.")
        return

    # 4. Visualization Setup (3 Subplots grouped by scale)
    fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=False, gridspec_kw={'height_ratios': [2.5, 1.5, 1]})
    sns.set_style("whitegrid")

    # Home Depot Color Scheme: Low Relevance = HD Primary Orange (#F96302), High Relevance = Slate Blue (#2B5C8F)
    palette = {
        'Low Relevance (≤ 2.33)': '#F96302',
        'High Relevance (> 2.33)': '#2B5C8F'
    }

    groups = ['Title Noise Ratio (%)', 'Description Sentiment (VADER)', 'Description Length (Chars)']

    for i, grp in enumerate(groups):
        ax = axes[i]
        grp_df = plot_df[plot_df['Metric Group'] == grp]

        if grp_df.empty:
            continue

        sns.barplot(
            data=grp_df,
            x='Department',
            y='Value',
            hue='Relevance Tier',
            ax=ax,
            palette=palette,
            edgecolor='black',
            linewidth=0.8,
            width=0.5
        )

        # Formatting per group
        ax.set_title(grp, fontsize=12, fontweight='bold', pad=10, loc='left')
        ax.set_xlabel('', fontweight='bold')
        ax.set_ylabel('Mean Value', fontweight='bold', fontsize=10)

        # Dynamic Y-Limits and Bar Annotations
        max_val = grp_df['Value'].max()
        ax.set_ylim(0, max_val * 1.3 if max_val > 0 else 1)

        unit_type = grp_df['Unit'].iloc[0]
        if unit_type == 'pct':
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

        # Add values directly above bars
        for p in ax.patches:
            height = p.get_height()
            if np.isnan(height) or height == 0:
                continue

            if unit_type == 'pct':
                label_text = f'{height:.1%}'
            elif unit_type == 'score':
                label_text = f'{height:.2f}'
            else:
                label_text = f'{height:,.0f}'

            ax.annotate(
                label_text,
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points',
                fontsize=9.5, fontweight='bold'
            )

        ax.legend(title='', frameon=True, facecolor='white', framealpha=0.9, loc='upper right')

    plt.suptitle('Relevance High/Low Split Analysis (Threshold = 2.33)', fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

# Run the split analysis on your DataFrame
plot_relevance_high_low_split(df)

In [ ]:
len(df.columns)